# Fig. 12, Guinness on a cart: every computation of the figure in one notebook

Phyein Usul (phyeinusul.com). Data: dataset `numakoudsie/fig12-guinness-on-a-cart` (attach it as input). Record of origin: `public/data/fig12/generations.json` in the site's repository. Built by `tools/kaggle/figures/build.py`.

Set `JOB` in the first code cell. `check` (default, CPU) verifies every stored file (size and SHA-256 against its manifest) and that every job's code compiles. Any other name reruns that job with the exact code that made the data; that code fetches its solver from the repository at the pinned commit. GPU jobs need the GPU switched on.

| JOB | computes | machine | commit | code and outputs in the dataset |
|---|---|---|---|---|
| `stout-in-jax` | the stout in JAX: the rides the page plays (current code is version 10) | GPU T4 x2 | `58eea68` | `guinness-on-a-cart-stout-in-jax/` |
| `stout-sloshing` | Taichi generation: the rides of rides.json (version 5) | GPU T4 x2 | `70703de` | `guinness-on-a-cart-stout-sloshing/` |
| `free-swing` | Taichi generation: the free swing and its repair (versions 4 and 5) | GPU T4 x2 | `3d6f61a` | `guinness-free-swing-v2/` |

In [ ]:
JOB = "check"   # or one of: stout-in-jax, stout-sloshing, free-swing
DATASET = "numakoudsie/fig12-guinness-on-a-cart"
JOBS = [
 {
  "name": "stout-in-jax",
  "folder": "guinness-on-a-cart-stout-in-jax",
  "machine": "GPU T4 x2",
  "commit": "58eea6870a20b8edfaa9ef46ee655cf1e890fdfe",
  "what": "the stout in JAX: the rides the page plays (current code is version 10)"
 },
 {
  "name": "stout-sloshing",
  "folder": "guinness-on-a-cart-stout-sloshing",
  "machine": "GPU T4 x2",
  "commit": "70703de1821fc5f28a5592a1a74c8d3c84a73488",
  "what": "Taichi generation: the rides of rides.json (version 5)"
 },
 {
  "name": "free-swing",
  "folder": "guinness-free-swing-v2",
  "machine": "GPU T4 x2",
  "commit": "3d6f61a79c7356fdb044c6f134cf385783a492b7",
  "what": "Taichi generation: the free swing and its repair (versions 4 and 5)"
 }
]


In [ ]:
# ---- find the figure's dataset, then check it or run one job ----
import os, json, glob, hashlib, time, datetime
T0 = time.time()
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/README.md", recursive=True)
         if all(os.path.isdir(os.path.join(os.path.dirname(p), j["folder"])) for j in JOBS)]
assert cands, "attach the dataset %s (Add Input) and run again" % DATASET
DS = cands[0]; print("dataset at", DS)
def source_of(job):
    nb = json.load(open(os.path.join(DS, job["folder"], "source-current-version.ipynb")))
    return [("".join(c["source"]) if isinstance(c["source"], list) else c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
OUT = "/kaggle/working/" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
os.makedirs(OUT, exist_ok=True)
if JOB == "check":
    from IPython.core.inputtransformer2 import TransformerManager
    TM = TransformerManager(); report = dict(dataset=DATASET, path=DS, jobs={}, versions={})
    for j in JOBS:
        cells = source_of(j)
        for n, src in enumerate(cells): compile(TM.transform_cell(src), f"{j['name']} cell {n}", "exec")
        assert any(j["commit"] in src for src in cells), f"job {j['name']}: the stored code does not name commit {j['commit']}"
        report["jobs"][j["name"]] = dict(folder=j["folder"], code_cells=len(cells), compiles=True, commit_in_code=True)
        print(f"job {j['name']:15s} code: {len(cells)} cells, all compile, commit {j['commit'][:7]} named in the code")
    for man in sorted(glob.glob(os.path.join(DS, "*", "v*", "manifest.json"))):
        m = json.load(open(man)); base = os.path.join(os.path.dirname(man), "output")
        same = differ = unpacked = dropped = 0; missing = []
        for f in m["files"]:
            p = os.path.join(base, f["file"])
            if os.path.isfile(p):
                h = hashlib.sha256()
                with open(p, "rb") as fh:
                    for blk in iter(lambda: fh.read(1 << 22), b""): h.update(blk)
                ok = os.path.getsize(p) == f["bytes"] and h.hexdigest() == f["sha256"]
                same += ok; differ += not ok
            elif f["file"].endswith(".tar") and (os.path.isdir(p) or os.path.isdir(p[:-4])): unpacked += 1   # Kaggle unpacks tars
            elif f["file"].endswith(".pyc"): dropped += 1                                                   # Kaggle drops __pycache__
            else: missing.append(f["file"])
        key = os.path.relpath(os.path.dirname(man), DS)
        report["versions"][key] = dict(files=len(m["files"]), identical=same, different=differ, tar_unpacked=unpacked, pyc_dropped=dropped, missing=missing)
        print(f"{key:45s} {len(m['files']):5d} files: {same} identical, {differ} different, {unpacked} tars unpacked, {dropped} .pyc dropped, {len(missing)} missing")
    report["ok"] = all(v["different"] == 0 and not v["missing"] for v in report["versions"].values())
    report["seconds"] = round(time.time() - T0, 1)
    json.dump(report, open(OUT + "/check.json", "w"), indent=1)
    print("ALL OK" if report["ok"] else "PROBLEMS FOUND", "in", report["seconds"], "s")
else:
    job = next(j for j in JOBS if j["name"] == JOB)
    print("running job", JOB, "with the code of", job["folder"], "(commit", job["commit"] + ")")
    for n, src in enumerate(source_of(job)):
        print(f"---- cell {n} ----\n{src}")
        r = get_ipython().run_cell(src)
        if not r.success: raise RuntimeError(f"cell {n} of job {JOB} failed")
